Lab 5: Linear Regression Neuron: Learning by Gradient
Descent

In [1]:
# Part A: Data Setup (you do this, no shortcuts)
# A1. Load dataset
import numpy as np
import pandas as pd
columns = [
    "Sex", "Length", "Diameter", "Height",
    "WholeWeight", "ShuckedWeight",
    "VisceraWeight", "ShellWeight", "Rings"
]

data = pd.read_csv("/content/abalone.data", header=None, names=columns)

print("Number of rows:", len(data))
print("Column names:", data.columns)
print(data.head())
# what is input: physical measurements of abalone
# what is output: age (derived from Rings)
# why output is numeric: age is a continuous numeric quantity

Number of rows: 4177
Column names: Index(['Sex', 'Length', 'Diameter', 'Height', 'WholeWeight', 'ShuckedWeight',
       'VisceraWeight', 'ShellWeight', 'Rings'],
      dtype='object')
  Sex  Length  Diameter  Height  WholeWeight  ShuckedWeight  VisceraWeight  \
0   M   0.455     0.365   0.095       0.5140         0.2245         0.1010   
1   M   0.350     0.265   0.090       0.2255         0.0995         0.0485   
2   F   0.530     0.420   0.135       0.6770         0.2565         0.1415   
3   M   0.440     0.365   0.125       0.5160         0.2155         0.1140   
4   I   0.330     0.255   0.080       0.2050         0.0895         0.0395   

   ShellWeight  Rings  
0        0.150     15  
1        0.070      7  
2        0.210      9  
3        0.155     10  
4        0.055      7  


Loads the dataset and displays basic information.

Why it is needed:
We must understand the structure of data before modeling.

In [2]:
# A2. Convert target
y = data["Rings"].values + 1.5


Converts ring count into estimated age.

Why it is needed:
The problem is numeric prediction (regression), not classification.

In [3]:
# A3. Choose features
X = data[["Length", "Diameter", "ShellWeight"]].values

# Feature 1: Length – overall size strongly correlates with age
# Feature 2: Diameter – complements length for body growth
# Feature 3: ShellWeight – older abalones have heavier shells


Linear neuron learns weights per feature; too many features complicate learning.

In [4]:
# A4. Train-test split
split_index = int(0.8 * len(X))

X_train = X[:split_index]
X_test = X[split_index:]

y_train = y[:split_index]
y_test = y[split_index:]

print(X_train.shape, y_train.shape)
print(X_test.shape, y_test.shape)


(3341, 3) (3341,)
(836, 3) (836,)


Separates data for learning and evaluation.

Why it is needed:
Testing on unseen data checks generalization.

In [6]:
# Normalize inputs
mean = X_train.mean(axis=0)
std = X_train.std(axis=0)
X_train = (X_train - mean) / std
X_test = (X_test - mean) / std

# why normalization is needed for learning:
# It prevents features with large scales from dominating gradients and stabilizes gradient descent.

In [7]:
# Define the model
def forward(X, w, b):
    y_hat = X.dot(w) + b
    print("X shape:", X.shape)
    print("w shape:", w.shape)
    print("b shape:", type(b))
    print("y_hat shape:", y_hat.shape)
    return y_hat

# parameters are: weights w and bias b
# number of parameters: d weights + 1 bias = 4 parameters


In [8]:
# Define Loss
def mse(y, y_hat):
    loss = np.mean((y - y_hat) ** 2)
    return loss

# why square: removes sign and penalizes large errors more
# what mistakes are expensive: very large prediction errors

In [9]:
# Learning Rule
# what gradient means in words:
# Gradient tells the direction and rate at which loss increases.

# why subtracting gradient reduces loss:
# Moving opposite to gradient moves parameters toward lower loss.

In [11]:
# Gradient Implementation
def grad_w(X, y, y_hat):
    N = len(y)
    dW = (-2 / N) * X.T.dot(y - y_hat)
    return dW

def grad_b(y, y_hat):
    N = len(y)
    db = (-2 / N) * np.sum(y - y_hat)
    return db

# meaning of large gradient: model is making large errors
# effect of too-large learning rate: updates overshoot and diverge

In [12]:
# Training Loop
np.random.seed(0)
w = np.random.randn(X_train.shape[1]) * 0.01
b = 0.0

lr = 0.01
epochs = 1000

for epoch in range(epochs):
    y_hat = X_train.dot(w) + b
    loss = mse(y_train, y_hat)

    dw = grad_w(X_train, y_train, y_hat)
    db = grad_b(y_train, y_hat)

    w -= lr * dw
    b -= lr * db

    if epoch % 100 == 0:
        print(f"Epoch {epoch}, Loss: {loss}")

# Initial expectation:
# Loss should decrease slowly due to simple linear model

# Revised expectation after training:
# Loss decreases steadily and stabilizes

Epoch 0, Loss: 144.17831736922406
Epoch 100, Loss: 9.201549523151352
Epoch 200, Loss: 6.791603706080492
Epoch 300, Loss: 6.6851394315379595
Epoch 400, Loss: 6.643926006132289
Epoch 500, Loss: 6.618599647121504
Epoch 600, Loss: 6.602358692230311
Epoch 700, Loss: 6.591561163764089
Epoch 800, Loss: 6.58405083878558
Epoch 900, Loss: 6.578546017935997


In [14]:
# Evaluation
y_test_hat = X_test.dot(w) + b
test_mse = mse(y_test, y_test_hat)

test_mae = np.mean(np.abs(y_test - y_test_hat))

print("Test MSE:", test_mse)
print("Test MAE:", test_mae)

# Print 5 example predictions
for i in range(5):
    print("True age:", y_test[i],
          "Predicted age:", y_test_hat[i],
          "Absolute error:", abs(y_test[i] - y_test_hat[i]))

# systematic errors: larger errors for very old abalones
# observed bias: model underestimates extreme ages

Test MSE: 5.11761055903468
Test MAE: 1.7243772416300003
True age: 13.5 Predicted age: 10.786728908057285 Absolute error: 2.7132710919427154
True age: 15.5 Predicted age: 9.651008710011215 Absolute error: 5.848991289988785
True age: 14.5 Predicted age: 10.061079010642514 Absolute error: 4.438920989357486
True age: 14.5 Predicted age: 11.119772007983864 Absolute error: 3.3802279920161364
True age: 13.5 Predicted age: 11.43350901583535 Absolute error: 2.06649098416465
